# Fetch Spatial Features — Road Density & POI Count (Overpass API)

Calcula **road length** y **POI density** para cada una de las 247 zonas del dataset ST-EVCDP usando la API **Overpass de OpenStreetMap** (gratuita, sin API key).

- Fuente: [overpass-api.de](https://overpass-api.de)
- Input: coordenadas `lon`/`la` de `information.csv`
- Método: radio de **1 km** alrededor del centroide de cada zona
- Output: `spatial_features_shenzhen.csv`

| Feature | Paper cita (§3.1.1) | Cómo se calcula |
|---|---|---|
| **Road length (km)** | `"Road Length = 83.23 km"` | Suma longitudes de `way["highway"]` en radio 1 km |
| **POI count** | `"POI density"` | Cuenta nodos `amenity + shop + leisure` en radio 1 km |

- Rate limit: ~1 req/s → ~8 min para 247 zonas
- Checkpointing: guarda progreso parcial cada 20 zonas

In [ ]:
!pip install tqdm requests -q

import requests
import pandas as pd
import numpy as np
import math
import time
import json
from pathlib import Path
from tqdm.notebook import tqdm

In [ ]:
# ── Rutas ─────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATASETS_PATH = "/content/drive/MyDrive/MASTER CD/2cuatri/Modelado computacional/Trabajo final/datasets"
    IN_COLAB = True
except ImportError:
    DATASETS_PATH = "./datasets"
    IN_COLAB = False

CHECKPOINT_PATH = "spatial_checkpoint.json"
OUTPUT_PATH     = "spatial_features_shenzhen.csv"

# ── Cargar information.csv ────────────────────────────────────────────────
info = pd.read_csv(f"{DATASETS_PATH}/information.csv", index_col=0)
info.columns = [c.strip().lower().replace(" ", "_") for c in info.columns]

LAT_COL = "la"
LON_COL = "lon"

print(f"Zonas: {len(info)}")
print(f"Lat range: {info[LAT_COL].min():.4f} – {info[LAT_COL].max():.4f}")
print(f"Lon range: {info[LON_COL].min():.4f} – {info[LON_COL].max():.4f}")

In [ ]:
# ── Fórmula Haversine para longitud de segmentos de carretera ─────────────
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Distancia en km entre dos coordenadas (fórmula haversine)."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))


def way_length_km(way: dict) -> float:
    """Calcula la longitud total (km) de una way de OSM a partir de su geometría."""
    geom = way.get("geometry", [])
    if len(geom) < 2:
        return 0.0
    total = 0.0
    for i in range(len(geom) - 1):
        total += haversine_km(geom[i]["lat"], geom[i]["lon"],
                              geom[i+1]["lat"], geom[i+1]["lon"])
    return total

In [ ]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
RADIUS_M     = 1000  # 1 km radio (igual que el paper)

# Tipos de highway incluidos (excluyendo footway/cycleway para road density)
HIGHWAY_TYPES = [
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "residential", "unclassified", "motorway_link", "trunk_link",
    "primary_link", "secondary_link", "tertiary_link"
]

def build_query(lat: float, lon: float, radius: int = RADIUS_M) -> str:
    highway_filter = "|".join(HIGHWAY_TYPES)
    return f"""
[out:json][timeout:60];
(
  way["highway"~"{highway_filter}"](around:{radius},{lat},{lon});
  node["amenity"](around:{radius},{lat},{lon});
  node["shop"](around:{radius},{lat},{lon});
  node["leisure"](around:{radius},{lat},{lon});
);
out body geom;
"""


def fetch_spatial(lat: float, lon: float, retries: int = 3) -> dict:
    """Fetch road length (km) y POI count para un punto con radio RADIUS_M."""
    query = build_query(lat, lon)
    for attempt in range(retries):
        try:
            resp = requests.post(OVERPASS_URL, data={"data": query}, timeout=90)
            if resp.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"    Rate limited — esperando {wait}s")
                time.sleep(wait)
                continue
            if resp.status_code == 504:
                # Server timeout — reducir radio y reintentar
                time.sleep(10)
                continue
            resp.raise_for_status()
            elements = resp.json().get("elements", [])

            road_km  = 0.0
            poi_count = 0

            for el in elements:
                if el["type"] == "way":
                    road_km += way_length_km(el)
                elif el["type"] == "node":
                    poi_count += 1

            return {
                "road_length_km": round(road_km, 3),
                "poi_count":      poi_count,
                "radius_m":       RADIUS_M,
                "error":          False,
            }
        except Exception as e:
            if attempt == retries - 1:
                return {"road_length_km": None, "poi_count": None,
                        "radius_m": RADIUS_M, "error": True}
            time.sleep(5 * (attempt + 1))

    return {"road_length_km": None, "poi_count": None, "radius_m": RADIUS_M, "error": True}


# Test con zona 0
sample_lat = float(info[LAT_COL].iloc[0])
sample_lon = float(info[LON_COL].iloc[0])
print(f"Test zona 0: lat={sample_lat}, lon={sample_lon}")
result = fetch_spatial(sample_lat, sample_lon)
print(f"Resultado: {result}")

In [ ]:
# ── Cargar checkpoint si existe ────────────────────────────────────────────
checkpoint = {}
if Path(CHECKPOINT_PATH).exists():
    with open(CHECKPOINT_PATH) as f:
        checkpoint = json.load(f)
    print(f"Checkpoint: {len(checkpoint)} zonas ya procesadas.")
else:
    print("Sin checkpoint — empezando desde zona 0.")

CHECKPOINT_EVERY = 20
SLEEP_BETWEEN   = 2.0  # Overpass: más conservador que Nominatim

results  = list(checkpoint.values()) if checkpoint else []
done_ids = set(checkpoint.keys())

zones_todo = [
    (i, row) for i, row in info.iterrows()
    if str(i) not in done_ids
]

print(f"Zonas restantes: {len(zones_todo)} / {len(info)}")
print(f"Tiempo estimado: ~{len(zones_todo) * SLEEP_BETWEEN / 60:.1f} min")

for zone_id, row in tqdm(zones_todo, desc="Overpass spatial"):
    lat = row.get(LAT_COL, None)
    lon = row.get(LON_COL, None)

    if pd.isna(lat) or pd.isna(lon):
        entry = {"zone_id": zone_id, "lat": None, "lon": None,
                 "road_length_km": None, "poi_count": None, "error": True}
    else:
        spatial = fetch_spatial(float(lat), float(lon))
        entry   = {"zone_id": zone_id, "lat": float(lat), "lon": float(lon), **spatial}

    results.append(entry)
    checkpoint[str(zone_id)] = entry

    if len(results) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(checkpoint, f)
        errors = sum(1 for r in results if r.get("error"))
        print(f"  Checkpoint guardado — {len(results)} zonas, {errors} errores")

    time.sleep(SLEEP_BETWEEN)

with open(CHECKPOINT_PATH, "w") as f:
    json.dump(checkpoint, f)

print(f"\nCompletado: {len(results)} zonas.")
print(f"Errores:    {sum(1 for r in results if r.get('error'))}")

In [ ]:
# ── Guardar CSV ────────────────────────────────────────────────────────────
df_spatial = pd.DataFrame(results).sort_values("zone_id").reset_index(drop=True)

# Imputar errores con mediana de zonas correctas
for col in ["road_length_km", "poi_count"]:
    median_val = df_spatial[col].median()
    n_missing  = df_spatial[col].isna().sum()
    df_spatial[col] = df_spatial[col].fillna(median_val)
    if n_missing:
        print(f"  {col}: {n_missing} valores imputados con mediana={median_val:.2f}")

# Densidades normalizadas [0,1] para uso directo en prompts
for col in ["road_length_km", "poi_count"]:
    mn, mx = df_spatial[col].min(), df_spatial[col].max()
    df_spatial[f"{col}_norm"] = ((df_spatial[col] - mn) / (mx - mn + 1e-8)).round(4)

df_spatial.drop(columns=["error", "radius_m"], errors="ignore", inplace=True)
df_spatial.to_csv(OUTPUT_PATH, index=False)

print(f"\nGuardado: {OUTPUT_PATH}")
print(f"Shape:    {df_spatial.shape}")
print(f"\nStats:")
print(df_spatial[["road_length_km", "poi_count"]].describe().round(2))

if IN_COLAB:
    import shutil
    dst = f"{DATASETS_PATH}/spatial_features_shenzhen.csv"
    shutil.copy(OUTPUT_PATH, dst)
    print(f"\nCopiado a Drive: {dst}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Spatial Features — 247 zonas Shenzhen (radio 1 km)", fontsize=12)

axes[0].hist(df_spatial["road_length_km"], bins=30, color="#E74C3C", edgecolor="white", alpha=0.85)
axes[0].axvline(df_spatial["road_length_km"].median(), color="black", lw=1.5, ls="--",
                label=f"Mediana: {df_spatial['road_length_km'].median():.1f} km")
axes[0].set_xlabel("Road length (km)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución Road Length")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].hist(df_spatial["poi_count"], bins=30, color="#2980B9", edgecolor="white", alpha=0.85)
axes[1].axvline(df_spatial["poi_count"].median(), color="black", lw=1.5, ls="--",
                label=f"Mediana: {df_spatial['poi_count'].median():.0f}")
axes[1].set_xlabel("POI count")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución POI Density")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

axes[2].scatter(df_spatial["lon"], df_spatial["lat"],
                c=df_spatial["poi_count"], cmap="YlOrRd",
                s=40, alpha=0.8, edgecolors="grey", linewidths=0.3)
sm = plt.cm.ScalarMappable(cmap="YlOrRd",
     norm=plt.Normalize(df_spatial["poi_count"].min(), df_spatial["poi_count"].max()))
plt.colorbar(sm, ax=axes[2], label="POI count")
axes[2].set_xlabel("Longitud")
axes[2].set_ylabel("Latitud")
axes[2].set_title("Mapa POI Density — Shenzhen")
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.savefig("spatial_features_preview.png", dpi=130, bbox_inches="tight")
print("Preview guardado: spatial_features_preview.png")
plt.show()

print("\nTop 5 zonas por road length:")
print(df_spatial.nlargest(5, "road_length_km")[["zone_id", "lat", "lon", "road_length_km", "poi_count"]].to_string())
print("\nTop 5 zonas por POI density:")
print(df_spatial.nlargest(5, "poi_count")[["zone_id", "lat", "lon", "road_length_km", "poi_count"]].to_string())